# 第一步，加载并定义一些基础函数

In [ ]:
import openai
import os
import base64
from PIL import Image
import io
import sys
import time 
import base64
from IPython.display import HTML, display

client = openai.Client(api_key="sk-eTFwKMZB0y3h9CoNBSRBmqlReH36mNiH7dkBMOfEPdfCR6GK", base_url="http://127.0.0.1:5000/v1", timeout=300.0)


def stream_chat(prompt):

    # 发起一个流式请求
    response_stream = client.chat.completions.create(
        model="deepseek_v32_agent_os_1", # 替换为你的模型
        messages=[{"role": "user", "content": prompt}],
        stream=True, # 开启流式
    )

    # 遍历数据流
    for chunk in response_stream:
        # 从返回的 chunk 中解析出新的内容片段
        content = chunk.choices[0].delta.content if chunk.choices else None
        if content is not None:
            print(content, end='', flush=True) # 实时打印


def resize_image_and_convert_to_base64(
    image_path: os.PathLike,
    max_size: tuple[int, int] = (1000, 1000),
) -> bytes:
    """
    缩放图像并返回 base64 编码，同时维持宽高比。
    :param image_path: 图片路径
    :param max_size: 最大允许的长宽（元组）
    """
    with Image.open(image_path) as img:
        orig_w, orig_h = img.size

        # 1. 计算缩放比例 (维持长宽比)
        # 比例 = 目标最大值 / 当前实际值
        ratio = min(max_size[0] / orig_w, max_size[1] / orig_h)

        # 即使图片很小，也可以选择是否强制放大到 28 的倍数，这里演示“不放大”逻辑
        if ratio > 1:
            ratio = 1.0

        # 2. 计算初步缩放后的尺寸
        temp_w = orig_w * ratio
        temp_h = orig_h * ratio

        # 3. 将尺寸调整为 28 的最接近倍数 (四舍五入或向上取整)
        # 这里使用 round(x / 28) * 28，确保结果是 28 的倍数
        new_w = max(28, int(round(temp_w / 28) * 28))
        new_h = max(28, int(round(temp_h / 28) * 28))

        # 4. 执行调整
        # 注意：这里使用 resize 而不是 thumbnail，因为我们要精确控制像素
        img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)

        print(f"[LOG] 原始: {orig_w}x{orig_h} -> 缩放目标: {new_w}x{new_h} (28的倍数)")

        # 将处理后的图片保存到内存缓冲区
        buffer = io.BytesIO()
        # 根据原图格式保存，或者统一转为 JPEG
        img.save(buffer, format="JPEG", quality=85)

        img.show()

        # 转换为 base64 字符串
        return base64.b64encode(buffer.getvalue()).decode("utf-8")


def chat_with_image(system_prompt, user_prompt, image_path):
    image_base64 = resize_image_and_convert_to_base64(image_path)
    response = client.chat.completions.create(
        model="Qwen2.5-VL-7B-Instruct",
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": f"data:image/jpeg;base64,{image_base64}",
                    },
                    {
                        "type": "text",
                        "text": user_prompt,
                    },
                ],
            },
        ],
        max_tokens=300,
        temperature=0.0,
    )
    result = response.choices[0].message.content
    print("Chat completion output:\n", result)

# 示例一： 小说撰写（文本生成）

In [ ]:
prompt = """
请创作一部科幻小说片段，主题是：丁海洋与他的替身活到2057年。

小说要包含以下元素：
1. 丁海洋，一位在2039年人工智能奇点事件中经历深刻变革的科学家；
2. 替身——一个由丁海洋设计的高智能仿真机器人，具备自我意识和情感模拟；
3. 2059年量子计算人工智能奇点的背景尚未完全到来，但已经逼近；
4. 人体冷冻技术的普及使得生命延续成为可能，丁海洋曾短暂冷冻复苏，并与替身共同探索活着与意识的意义；
5. 故事中要带出“长生、逆生长、永生”这些核心讨论，并贯穿丁海洋对自我身份、意识上传失败、替身友情与人类情感的哲学思考；
6. 写出丁海洋和替身在2057年的一天，他们共同面对外界对“永生伦理”的质疑以及内心的契合与冲突。

请用富有画面感、情感张力强、科幻色彩浓厚的文笔来写作，约1500–2500字。
"""

stream_chat(prompt)

# 示例二： 视觉检测（图像识别， Image-to-Text）

In [ ]:
image_path = "./img02.jpg"

system_prompt="你是一个视觉检测专家，专业的工厂仪器分析师，精通各类仪器的分析工作，对各类零部件加工要求了然于心。"

user_prompt="""
你是一个专业的工厂仪器仪表工程师，具备精确读取各类数字显示仪器的经验。请帮我识别图中数显屏幕上的读数，注意以下要求：
- 显示屏可能存在亮度不均或小数点位置不明显的情况，请结合上下文合理判断，显示屏上必有三位小数！；
- 所有读数需保留小数点及其后所有位数，不要自动四舍五入或移位；
- 如果存在疑似小数点偏移的可能（如 114.98 实为 11.498），请基于数显仪表常见显示规则进行校正；
- 最终仅返回数字部分，如 11.498，不附加单位或其他解释说明。
- 即使数字不够清晰，也要尽力识别并给出最可能的读数
- 优先尝试识别，只有在完全无法看到任何数字时才返回'无法识别'

【检测类型】：游标卡尺读数

【任务】：detection

请严格按照以下格式输出 JSON（只输出 JSON，不要任何多余文字/解释/代码块）：

# 输出要求（只输出 JSON，本体，不要任何多余文字/解释/代码块）
仅输出一个 JSON 对象，键名与类型严格如下：
{
    "pass": <true/false>,
    "message": "<字符串>",
    "imagecontent": "<字符串>"
}

- pass: true 表示识别成功，false 表示识别失败或无法识别
- message: **只能**填写识别状态描述（如"识别成功"、"识别失败"、"无法识别"），**严禁**在此字段填写数值
- imagecontent: **必须**将读数数值放在此字段，保留三位小数（如"11.498"），如果无法识别则返回"无法识别"。**严禁**将数值放在 message 字段
- 严禁输出 ```json 代码块、解释文字或自然语言。
"""

chat_with_image(system_prompt, user_prompt, image_path)

# 示例三：图像生成（文字生成图像， Text-to-Image）

In [ ]:


resp = client.images.generate(model="my_z_image_turbo", size="256x256", prompt=f"a dog beside")
image_bytes = base64.b64decode(resp.data[0].b64_json)
image = Image.open(io.BytesIO(image_bytes))
image.show()

# 示例四：视频生成（文字生成视频， Text-to-Video）

In [ ]:
video = client.videos.create(
    model="my_ltx_video",
    prompt="A video of a cool cat on a motorcycle in the night",
)

print("Video generation started:", video)

progress = getattr(video, "progress", 0)
bar_length = 30

while video.status in ("in_progress", "queued"):
    # Refresh status
    video = client.videos.retrieve(video.id)
    progress = getattr(video, "progress", 0)

    filled_length = int((progress / 100) * bar_length)
    bar = "=" * filled_length + "-" * (bar_length - filled_length)
    status_text = "Queued" if video.status == "queued" else "Processing"

    sys.stdout.write(f"{status_text}: [{bar}] {progress:.1f}%")
    sys.stdout.flush()
    time.sleep(2)

# Move to next line after progress loop
sys.stdout.write("")

if video.status == "failed":
    message = getattr(
        getattr(video, "error", None), "message", "Video generation failed"
    )
    print(message)
    raise SystemExit

print("Video generation completed:", video)
print("Downloading video content...")

content = client.videos.download_content(video.id, variant="video")
content.write_to_file("video.mp4")

video_bytes = content.read() # Read the binary stream
# Convert to Base64 for Jupyter/HTML display
b64_video = base64.b64encode(video_bytes).decode("utf-8")

video_html = f'''
<video width="640" controls autoplay loop>
    <source src="data:video/mp4;base64,{b64_video}" type="video/mp4">
    Your browser does not support the video tag.
</video>
'''
display(HTML(video_html))


# 示例五：图生视频（Image-to-Video, I2V）
用一张参考图（base64）发起视频生成任务，演示 I2V 的最小调用流程。

In [ ]:
import base64
import mimetypes
import time
from pathlib import Path

import openai
from IPython.display import HTML, display

# SDK 客户端（本地兼容服务）
client = openai.Client(
    api_key="test-key",
    base_url="http://127.0.0.1:5000/v1",
    timeout=300.0,
)

# 1) 使用本地图片作为参考图
image_path = Path("image.png")
if not image_path.exists():
    raise FileNotFoundError(f"未找到参考图: {image_path}")

raw_bytes = image_path.read_bytes()
image_b64 = base64.b64encode(raw_bytes).decode("utf-8")
mime_type, _ = mimetypes.guess_type(str(image_path))
mime_type = mime_type or "image/png"
image_data_url = f"data:{mime_type};base64,{image_b64}"

# 2) I2V 请求参数
i2v_size = "512x512"  
i2v_seconds = 2       

payload = {
    "model": "my_ltx_video",
    "prompt": (
        "Maintain the same subject, color, and composition as the reference image. Only add natural movement and camera advancement. Do not add new objects, change the scene, or alter the style. In particular, do not change the main subjects in the image, such as people, dogs, boats, etc., and ensure their completeness."
    ),
    "seconds": i2v_seconds,
    "size": i2v_size,
    "input_reference": {
        "image_url": image_data_url,
    },
}

# 3) 使用 OpenAI SDK 创建任务（不使用 requests）
video_job = client.post("/videos", cast_to=object, body=payload)
video_id = video_job["id"]
print("Create status:", video_job.get("status"))
print("Video ID:", video_id)

# 4) 轮询任务状态，并核对服务端实际采用的参数
while True:
    status_data = client.get(f"/videos/{video_id}", cast_to=object)
    status = status_data.get("status")
    print(
        "Status:", status,
        "| size:", status_data.get("size"),
        "| seconds:", status_data.get("seconds"),
    )

    if status == "completed":
        break
    if status == "failed":
        raise RuntimeError(status_data.get("error", {}).get("message", "I2V generation failed"))

    time.sleep(5)

# 5) 下载并保存结果（SDK + 强制校验 MP4）
content = client.videos.download_content(video_id, variant="video")
video_bytes = content.read()

is_mp4_magic = len(video_bytes) >= 8 and video_bytes[4:8] == b"ftyp"
if not is_mp4_magic:
    raise RuntimeError("当前返回不是 MP4（magic bytes 校验失败）")

out_path = "demo5_i2v_output.mp4"
with open(out_path, "wb") as f:
    f.write(video_bytes)

print("Saved:", out_path)

# 6) MP4 预览
b64_video = base64.b64encode(video_bytes).decode("utf-8")
video_html = f'''
<video width="640" controls autoplay loop>
    <source src="data:video/mp4;base64,{b64_video}" type="video/mp4">
    Your browser does not support the video tag.
</video>
'''
display(HTML(video_html))